In [1]:
import polars as pl
import pickle

In [2]:
with open('../../data/raw/eth.pkl', 'rb') as f:
    eth = pickle.load(f)

In [3]:
edge_data = [
    (u, v, d['amount'], d['timestamp'])
    for u, v, k, d in eth.edges(keys=True, data=True)
]

df_edges = pl.DataFrame(edge_data, schema=['from', 'to', 'amount', 'timestamp'], orient='row')

In [4]:
df_nodes = pl.DataFrame([
    (n, d.get('isp', None)) for n, d in eth.nodes(data=True)
], schema=['account', 'isp'], orient='row')

In [5]:
mapping_node_to_id = (
    df_nodes.select(
        pl.col('account')
    )
    .unique()
    .with_row_index('node_id')
)
mapping_node_to_id.head()

node_id,account
u32,str
0,"""0xae1de420b7ec6f63aff81c2497bb…"
1,"""0x4d85686a318e317be8a3dcaace9d…"
2,"""0x50fd9318913e94c17ad422aa65c7…"
3,"""0x048fd8349a79d9abe0524d657459…"
4,"""0x5b8d04eac1ecc08efd66a6d286bd…"


In [6]:
df_nodes_with_ids = (
    df_nodes.join(mapping_node_to_id, on='account')
    .drop('account')
)
df_nodes_with_ids.head()

isp,node_id
i64,u32
0,0
0,1
0,2
0,3
0,4


In [7]:
df_edges_with_ids = (
    df_edges.join(mapping_node_to_id, left_on='from', right_on='account')
    .drop('from')
    .rename({
        'node_id': 'from'
    })
    .join(mapping_node_to_id, left_on='to', right_on='account')
    .drop('to')
    .rename({
        'node_id': 'to'
    })
    .with_columns(
        pl.col('timestamp').sub(pl.col('timestamp').min()).add(10).cast(pl.Int64).alias('timestamp')
    )
    .sort('timestamp')
)
df_edges_with_ids.head()

amount,timestamp,from,to
f64,i64,u32,u32
105.9895,10,1416658,1586268
5999.5895,34,2827639,1586268
30.45825,36,973817,1586268
0.9895,57,1528111,1586268
9.8895,59,1053172,1586268


In [8]:
first_timestamp_node = (
    df_edges_with_ids.select(['timestamp', 'from']).rename({'from': 'node_id'})
    .vstack(
        df_edges_with_ids.select(['timestamp', 'to']).rename({'to': 'node_id'})
    )
    .group_by('node_id')
    .agg(
        pl.col('timestamp').min().alias('first_timestamp')
    )
)

df_nodes_with_ids_timestamps = df_nodes_with_ids.join(first_timestamp_node, on='node_id')

df_nodes_with_ids_timestamps.head()

isp,node_id,first_timestamp
i64,u32,i64
0,1368745,65940169
0,1155732,92044394
0,1978285,88236150
0,1627260,67370024
0,1168478,90895098


In [9]:
assert len(df_edges_with_ids) == 13551303

In [10]:
assert len(df_nodes_with_ids_timestamps) == 2973489

In [11]:
df_nodes_with_ids_timestamps.write_csv('../../data/raw/eth_nodes.csv')
df_edges_with_ids.write_csv('../../data/raw/eth_edges.csv')